# Faraday Rotation Analysis — Plots

Load simulation results from `faraday_sims.ipynb` and produce:
1. **Waterfall plots** (frequency vs time) of pseudo-Stokes I, Q, U — with/without FR, narrow/wide bins
2. **Delay-space waterfall plots** (delay vs time) — same parameter space
3. **Time-averaged spectra and delay profiles** — galaxy-up vs galaxy-down windows
4. All at 10, 30, and 50 MHz

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from lunarsky import Time

%matplotlib widget

RES_DIR = Path("/home/christian/Documents/research/lusee/lusee_faraday/notebooks/results")
CENTER_FREQS = [10, 30, 50]

In [ ]:
# Load all simulation results
data = {}
for cf in CENTER_FREQS:
    d = dict(np.load(RES_DIR / f"faraday_sim_{cf}mhz.npz"))
    data[cf] = d
    print(f"{cf} MHz: {sorted(d.keys())}")

# Reconstruct Time objects for axis labels
times_jd = data[CENTER_FREQS[0]]["times_jd"]
times_dt = [Time(jd, format="jd").to_datetime() for jd in times_jd]
ntimes = len(times_dt)

In [ ]:
# --- Identify galaxy-up and galaxy-down time windows ---
# Use Stokes I (no-FR wide) as proxy: high I = galaxy above horizon
# Pick the center frequency (30 MHz) as reference
ref_I = data[30]["pI_noFR_wide"].mean(axis=1)  # mean over freq bins
threshold = np.median(ref_I)
gal_up = ref_I > threshold
gal_down = ~gal_up
print(f"Galaxy up: {gal_up.sum()} time steps, "
      f"Galaxy down: {gal_down.sum()} time steps")

# Quick check
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(times_dt, ref_I, "k-", lw=0.8)
ax.axhline(threshold, color="r", ls="--", lw=0.8, label="threshold")
ax.fill_between(times_dt, ref_I.min(), ref_I.max(),
                where=gal_up, alpha=0.15, color="orange", label="galaxy up")
ax.set_ylabel("Mean Stokes I (30 MHz)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Plotting Helpers

In [ ]:
def waterfall_IQU(freqs, pI, pQ, pU, suptitle="", freq_label="Frequency [MHz]",
                   vmin_I=None, vmax_I=None, vmin_QU=None, vmax_QU=None):
    """3-panel waterfall plot for Stokes I, Q, U vs frequency and time."""
    extent = [freqs[0], freqs[-1], times_dt[-1], times_dt[0]]
    fig, axs = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True,
                            constrained_layout=True)
    for ax, arr, label, vmin, vmax in zip(
        axs, [pI, pQ, pU], ["Stokes I", "Stokes Q", "Stokes U"],
        [vmin_I, vmin_QU, vmin_QU], [vmax_I, vmax_QU, vmax_QU],
    ):
        im = ax.imshow(arr, aspect="auto", extent=extent, vmin=vmin, vmax=vmax)
        ax.set_title(label)
        ax.set_xlabel(freq_label)
        ax.yaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    axs[0].set_ylabel("Time")
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    return fig


def delay_waterfall_IQU(freqs, pI, pQ, pU, suptitle="",
                         vmin=None, vmax=None, log=True):
    """3-panel waterfall of delay transform (FT along freq axis)."""
    df = freqs[1] - freqs[0]  # MHz
    tau = np.fft.rfftfreq(len(freqs), d=df)  # 1/MHz = us
    extent = [tau[0], tau[-1], times_dt[-1], times_dt[0]]

    fig, axs = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True,
                            constrained_layout=True)
    for ax, arr, label in zip(axs, [pI, pQ, pU],
                               ["Stokes I", "Stokes Q", "Stokes U"]):
        ft = np.fft.rfft(arr, axis=1)
        vals = np.log10(np.abs(ft)) if log else np.abs(ft)
        im = ax.imshow(vals, aspect="auto", extent=extent, vmin=vmin, vmax=vmax)
        ax.set_title(label)
        ax.set_xlabel(r"Delay [$\mu$s]")
        ax.yaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    axs[0].set_ylabel("Time")
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    return fig


def plot_time_avg(freqs, noFR, FR, gal_up, gal_down, stokes_label="I",
                  suptitle="", freq_label="Frequency [MHz]"):
    """2-panel plot: galaxy-up and galaxy-down time-averaged spectra.
    noFR, FR: (ntimes, nfreq) arrays.
    """
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True,
                            constrained_layout=True)
    for ax, mask, title in zip(axs, [gal_up, gal_down],
                                ["Galaxy Up", "Galaxy Down"]):
        avg_noFR = noFR[mask].mean(axis=0)
        avg_FR = FR[mask].mean(axis=0)
        ax.plot(freqs, avg_noFR, label="No FR", lw=1)
        ax.plot(freqs, avg_FR, label="FR", lw=1, ls="--")
        ax.set_title(title)
        ax.set_xlabel(freq_label)
        ax.legend(fontsize=8)
    axs[0].set_ylabel(f"Stokes {stokes_label} [K]")
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    return fig


def plot_delay_avg(freqs, noFR, FR, gal_up, gal_down, stokes_label="I",
                   suptitle="", log=True):
    """2-panel delay-space plot: galaxy-up and galaxy-down averages."""
    df = freqs[1] - freqs[0]
    tau = np.fft.rfftfreq(len(freqs), d=df)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True,
                            constrained_layout=True)
    for ax, mask, title in zip(axs, [gal_up, gal_down],
                                ["Galaxy Up", "Galaxy Down"]):
        ft_noFR = np.fft.rfft(noFR[mask], axis=1)
        ft_FR = np.fft.rfft(FR[mask], axis=1)
        avg_noFR = np.abs(ft_noFR).mean(axis=0)
        avg_FR = np.abs(ft_FR).mean(axis=0)
        if log:
            ax.semilogy(tau, avg_noFR, label="No FR", lw=1)
            ax.semilogy(tau, avg_FR, label="FR", lw=1, ls="--")
        else:
            ax.plot(tau, avg_noFR, label="No FR", lw=1)
            ax.plot(tau, avg_FR, label="FR", lw=1, ls="--")
        ax.set_title(title)
        ax.set_xlabel(r"Delay [$\mu$s]")
        ax.legend(fontsize=8)
    axs[0].set_ylabel(f"|FT(Stokes {stokes_label})|")
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    return fig

## Waterfall Plots — Narrow (Zoom) Bins

For each center frequency: I, Q, U waterfalls with and without FR, plus the
FR-induced difference, in both frequency and delay space.

In [ ]:
for cf in CENTER_FREQS:
    d = data[cf]
    fz = d["freqs_zoom"]

    # No-FR waterfall (frequency space)
    waterfall_IQU(
        fz, d["pI_noFR_zoom"], d["pQ_noFR_zoom"], d["pU_noFR_zoom"],
        suptitle=f"No FR — Narrow bins @ {cf} MHz",
    )
    plt.show()

    # FR waterfall (frequency space)
    waterfall_IQU(
        fz, d["pI_FR_zoom"], d["pQ_FR_zoom"], d["pU_FR_zoom"],
        suptitle=f"FR — Narrow bins @ {cf} MHz",
    )
    plt.show()

    # Difference (FR - noFR)
    dI = d["pI_FR_zoom"] - d["pI_noFR_zoom"]
    dQ = d["pQ_FR_zoom"] - d["pQ_noFR_zoom"]
    dU = d["pU_FR_zoom"] - d["pU_noFR_zoom"]
    waterfall_IQU(
        fz, dI, dQ, dU,
        suptitle=f"FR effect (FR - noFR) — Narrow @ {cf} MHz",
    )
    plt.show()

    # Delay waterfalls
    delay_waterfall_IQU(
        fz, d["pI_noFR_zoom"], d["pQ_noFR_zoom"], d["pU_noFR_zoom"],
        suptitle=f"No FR — Narrow delay @ {cf} MHz",
    )
    plt.show()

    delay_waterfall_IQU(
        fz, d["pI_FR_zoom"], d["pQ_FR_zoom"], d["pU_FR_zoom"],
        suptitle=f"FR — Narrow delay @ {cf} MHz",
    )
    plt.show()

## Waterfall Plots — Wide Bins

In [ ]:
for cf in CENTER_FREQS:
    d = data[cf]
    fw = d["freqs_wide"]

    # Use convolved versions when available
    pI_noFR = d.get("pI_noFR_wconv", d["pI_noFR_wide"])
    pQ_noFR = d.get("pQ_noFR_wconv", d["pQ_noFR_wide"])
    pU_noFR = d.get("pU_noFR_wconv", d["pU_noFR_wide"])
    pI_FR = d.get("pI_FR_wconv", d["pI_FR_wide"])
    pQ_FR = d.get("pQ_FR_wconv", d["pQ_FR_wide"])
    pU_FR = d.get("pU_FR_wconv", d["pU_FR_wide"])

    waterfall_IQU(
        fw, pI_noFR, pQ_noFR, pU_noFR,
        suptitle=f"No FR — Wide bins @ {cf} MHz",
    )
    plt.show()

    waterfall_IQU(
        fw, pI_FR, pQ_FR, pU_FR,
        suptitle=f"FR — Wide bins @ {cf} MHz",
    )
    plt.show()

    dI = pI_FR - pI_noFR
    dQ = pQ_FR - pQ_noFR
    dU = pU_FR - pU_noFR
    waterfall_IQU(
        fw, dI, dQ, dU,
        suptitle=f"FR effect (FR - noFR) — Wide @ {cf} MHz",
    )
    plt.show()

    # Delay waterfalls
    delay_waterfall_IQU(
        fw, pI_noFR, pQ_noFR, pU_noFR,
        suptitle=f"No FR — Wide delay @ {cf} MHz",
    )
    plt.show()

    delay_waterfall_IQU(
        fw, pI_FR, pQ_FR, pU_FR,
        suptitle=f"FR — Wide delay @ {cf} MHz",
    )
    plt.show()

## Time-Averaged Spectra — Galaxy Up vs Down

Compare FR and no-FR in frequency space and delay space, averaged over
galaxy-up and galaxy-down time windows.

In [ ]:
for cf in CENTER_FREQS:
    d = data[cf]

    # --- Narrow bins: frequency-space averages ---
    fz = d["freqs_zoom"]
    for slabel, noFR_key, FR_key in [
        ("I", "pI_noFR_zoom", "pI_FR_zoom"),
        ("Q", "pQ_noFR_zoom", "pQ_FR_zoom"),
        ("U", "pU_noFR_zoom", "pU_FR_zoom"),
    ]:
        plot_time_avg(
            fz, d[noFR_key], d[FR_key], gal_up, gal_down,
            stokes_label=slabel,
            suptitle=f"Narrow @ {cf} MHz — Stokes {slabel}",
        )
        plt.show()

    # --- Narrow bins: delay-space averages ---
    for slabel, noFR_key, FR_key in [
        ("I", "pI_noFR_zoom", "pI_FR_zoom"),
        ("Q", "pQ_noFR_zoom", "pQ_FR_zoom"),
        ("U", "pU_noFR_zoom", "pU_FR_zoom"),
    ]:
        plot_delay_avg(
            fz, d[noFR_key], d[FR_key], gal_up, gal_down,
            stokes_label=slabel,
            suptitle=f"Narrow delay @ {cf} MHz — Stokes {slabel}",
        )
        plt.show()

    # --- Wide bins: use convolved versions ---
    fw = d["freqs_wide"]
    pI_noFR = d.get("pI_noFR_wconv", d["pI_noFR_wide"])
    pQ_noFR = d.get("pQ_noFR_wconv", d["pQ_noFR_wide"])
    pU_noFR = d.get("pU_noFR_wconv", d["pU_noFR_wide"])
    pI_FR = d.get("pI_FR_wconv", d["pI_FR_wide"])
    pQ_FR = d.get("pQ_FR_wconv", d["pQ_FR_wide"])
    pU_FR = d.get("pU_FR_wconv", d["pU_FR_wide"])

    for slabel, noFR_arr, FR_arr in [
        ("I", pI_noFR, pI_FR),
        ("Q", pQ_noFR, pQ_FR),
        ("U", pU_noFR, pU_FR),
    ]:
        plot_time_avg(
            fw, noFR_arr, FR_arr, gal_up, gal_down,
            stokes_label=slabel,
            suptitle=f"Wide @ {cf} MHz — Stokes {slabel}",
        )
        plt.show()

    # --- Wide bins: delay-space averages ---
    for slabel, noFR_arr, FR_arr in [
        ("I", pI_noFR, pI_FR),
        ("Q", pQ_noFR, pQ_FR),
        ("U", pU_noFR, pU_FR),
    ]:
        plot_delay_avg(
            fw, noFR_arr, FR_arr, gal_up, gal_down,
            stokes_label=slabel,
            suptitle=f"Wide delay @ {cf} MHz — Stokes {slabel}",
        )
        plt.show()

## Fractional FR Effect — Summary

For each Stokes parameter and mode, show the fractional difference
|FR - noFR| / |noFR| averaged over all times. This summarizes the
magnitude of the Faraday signal as a function of frequency across all
three center frequencies.

In [ ]:
# Summary: fractional FR effect for Q and U
fig, axs = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)

for col, (mode, fkey, noFR_sfx, FR_sfx) in enumerate([
    ("Narrow (zoom)", "freqs_zoom", "_noFR_zoom", "_FR_zoom"),
    ("Wide (25 kHz)", "freqs_wide", "_noFR_wide", "_FR_wide"),
]):
    for row, slabel in enumerate(["Q", "U"]):
        ax = axs[row, col]
        for cf in CENTER_FREQS:
            d = data[cf]
            f = d[fkey]
            noFR = d[f"p{slabel}{noFR_sfx}"]
            FR = d[f"p{slabel}{FR_sfx}"]
            # RMS of difference relative to RMS of noFR, per frequency bin
            diff_rms = np.sqrt(np.mean((FR - noFR) ** 2, axis=0))
            noFR_rms = np.sqrt(np.mean(noFR ** 2, axis=0))
            frac = np.where(noFR_rms > 0, diff_rms / noFR_rms, 0)
            ax.plot(f, frac * 100, label=f"{cf} MHz", lw=1)
        ax.set_ylabel(f"Stokes {slabel}: RMS(FR-noFR)/RMS(noFR) [%]")
        ax.set_xlabel("Frequency [MHz]")
        ax.legend(fontsize=8)
        if row == 0:
            ax.set_title(mode)

fig.suptitle("Fractional Faraday Rotation Effect", fontsize=13)
plt.show()